In [9]:
from google.colab import drive

drive.mount('/content/drive')
%cd /content/drive/MyDrive

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive


In [10]:
import os

repo_path = "/content/drive/MyDrive/bitenet"

if os.path.isdir(repo_path):
    %cd /content/drive/MyDrive/bitenet
    !git fetch origin
    !git reset --hard origin/fix/colab_notebook
else:
    %cd /content/drive/MyDrive
    !git clone -b fix/colab_notebook https://github.com/agataben/bitenet.git
    %cd bitenet

/content/drive/MyDrive/bitenet
HEAD is now at b46da851 Add training notebook


In [11]:
%pip install -r "colab_requirements.txt"

In [12]:
from src.utils import set_seed

seed = 1238
set_seed(seed)

In [13]:
from src.utils import get_norm_parameters

yaml_path = 'data'
mean, std = get_norm_parameters(yaml_path = yaml_path)
print(mean)
print(std)

tensor([0.5407, 0.4469, 0.3528])
tensor([0.2724, 0.2766, 0.2815])


In [14]:
from torchvision import transforms

train_transf = transforms.Compose([ transforms.Resize(256),
                                    transforms.CenterCrop(224),
                                    transforms.ToTensor(),
                                    transforms.Normalize(mean,std)
                                  ])

val_transf = transforms.Compose([ transforms.Resize(256),
                                  transforms.CenterCrop(224),
                                  transforms.ToTensor(),
                                  transforms.Normalize(mean,std)
                                ])


In [15]:
from src.food101_dataset import Food101DataSet

data_root = 'data'
train_ds = Food101DataSet(data_root = data_root, csv = 'data/train.csv', transform = train_transf)
val_ds = Food101DataSet(data_root = data_root, csv = 'data/val.csv', transform = val_transf)

In [16]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_ds, batch_size = 1024, num_workers = 2, shuffle = True)
val_loader = DataLoader(val_ds, batch_size = 1024, num_workers = 2, shuffle = False)
loaders = {'train': train_loader,
         'test': val_loader}

In [17]:
from src.bitenet_v1 import BiteNetV1

model = BiteNetV1()

In [20]:
%load_ext tensorboard
%tensorboard --logdir "/content/drive/MyDrive/bitenet/results/bitenet_v1/logs/exp_1"

<IPython.core.display.Javascript object>

In [18]:
import torch

print(torch.cuda.is_available())

True


In [19]:
from src.training import train

exp_name = 'exp_1'
ckpt_dir = '/content/drive/MyDrive/bitenet/results/bitenet_v1/ckpt'
logdir = '/content/drive/MyDrive/bitenet/results/bitenet_v1/logs'
model = train(model, loaders, lr = 0.01, momentum = 0.99, epochs = 1,
              exp_name = exp_name, logdir = logdir, ckpt_dir = ckpt_dir)